# JOINTS 2026 Main Notebook

Objective: Memprediksi penjualan tiket harian D4-D10 untuk setiap pasangan film dan klaster bioskop berdasarkan penjualan D1-D3 dengan menggunakan MASE sebagai metrik evaluasi.

# Import modules dan libraries

In [1]:
import os, glob, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import sdm
from sdm import TableTensor
from tabfm import TabFMRegressor, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

# Import datasets

In [2]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

DATA_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\data


### External datasets
Calendar files built by external_add.ipynb in the external/ folder.

In [3]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

EXTERNAL_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\external


# Aggregate Dataset
Menggabungkan semua file menjadi satu dataset utuh dengan satu baris per film, bioskop, dan tanggal target (D4-D10), sama seperti format test.csv. Train dan test digabung dalam satu tabel dengan penanda is_train, dan semua kolom mentah dari holidays, harga tiket, dan metadata film ikut digabung tanpa diolah.

### Penentuan D1
D1 train adalah tanggal pertama film tayang di minimal 50% dari jumlah bioskop maksimalnya, agar hari preview tidak dihitung sebagai D1. D1 test adalah tanggal pertama film di test_history. Film train yang sudah tayang sejak awal data atau belum mencapai D10 sebelum data berakhir dibuang.

In [4]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

film train: 205 | film test_history: 163 | film test: 160


### Pasangan film-bioskop
Mengambil transaksi D1-D10 (train) atau D1-D3 (test) setiap pasangan film dan bioskop, lalu menjadikan setiap hari sebagai kolom dan mengisi hari tanpa transaksi dengan 0. Pasangan train hanya diambil jika masih ada penjualan di D3, mengikuti test yang hanya berisi pasangan yang masih tayang di D3.

In [5]:
DAILY_COLS = {"total_ticket": "t", "occupation_rate": "occ", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

pasangan train: 8007 | pasangan test: 10373


### Format harian
Mengubah setiap pasangan menjadi 7 baris D4-D10, sehingga satu baris mewakili satu tanggal yang diprediksi. Input yang dibawa hanya D1-D3 (tiket, okupansi, jumlah tayang), dan total_ticket adalah target yang kosong untuk test.

In [6]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

baris train: 56049 | baris test: 72611 | baris test.csv: 72611


### Penggabungan semua sumber data
Menambahkan kolom mentah dari sumber lain ke dataset:
- city_name: kota tempat bioskop berada.
- day_tipe, holiday_tipe, holiday_name: tipe hari dan status libur tanggal target dari holidays.
- price: harga tiket kota untuk tipe hari tanggal target dari ticket_prices.
- age_rating, genre, producer, director, writer, casts: metadata film dari movies, dicocokkan lewat judul tanpa akhiran format (3D, IMAX).

In [7]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

full["city_name"] = full["cinema_ids"].map(CINEMA_CITY)
full = full.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
full["price_day"] = full["day_tipe"].map(PRICE_DAY)
full = full.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
full["base_title"] = full["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
full = full.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")

assert len(full) == len(train_long) + len(test_long)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

full: (128660, 30)


total_ticket     72611
id               56049
holiday_name    122732
dtype: int64

### External datasets join
Adds the raw columns of the external calendar files on the target date, left empty on dates outside them:
- cuti_bersama_name: the holiday a collective leave day belongs to.
- ramadan_day: day number of the fasting month (1-30).
- school_break: the school break period (DKI Jakarta).

In [8]:
full = full.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
full = full.merge(ramadan[["date", "ramadan_day"]].rename(columns={"date": "date_show"}), on="date_show", how="left")
full = full.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")

assert len(full) == len(train_long) + len(test_long)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "ramadan_day", "school_break"]].count()

full: (128660, 33)


,cuti_bersama_name,ramadan_day,school_break
is_train,,,
0,2392,12729,12540
1,1141,0,6208


In [9]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,occ1,occ2,occ3,show1,...,base_title,age_rating,genre,producer,director,writer,casts,cuti_bersama_name,ramadan_day,school_break
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,13.56,14.30,12.25,18.0,...,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN,NaN
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,39.38,23.87,22.75,23.0,...,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN,NaN
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.00,16.10,16.14,0.0,...,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN,NaN
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,19.57,38.15,43.26,9.0,...,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN,NaN
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,12.21,14.42,15.14,30.0,...,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN,NaN


# Preprocessing
Cleans the aggregated dataset: removes broken training rows, turns raw columns into model-ready values, drops columns without usable signal, and builds the target.

### Drop incomplete days
Days in train with far fewer rows than the 15-day median are recording gaps (7-16 June 2025), not real drops in demand. Training rows whose target date is a gap day get a fake zero, and rows whose D1-D3 touches a gap day get a wrong scale, so both are removed.

In [10]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

d13_dates = [full["d1_date"] + pd.Timedelta(days=k) for k in range(3)]
target_on_gap = full["date_show"].isin(GAP_DAYS)
d13_on_gap = np.any([d.isin(GAP_DAYS) for d in d13_dates], axis=0)
broken = (full["is_train"] == 1) & (target_on_gap | d13_on_gap)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
print(f"dropped train rows: {broken.sum()} (target on gap: {(broken & target_on_gap).sum()}, D1-D3 on gap: {(broken & d13_on_gap).sum()})")
full = full[~broken].reset_index(drop=True)
print(f"full: {full.shape}")

gap days: ['2025-06-07', '2025-06-09', '2025-06-11', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16']
dropped train rows: 1378 (target on gap: 601, D1-D3 on gap: 959)
full: (127282, 33)


### Calendar flags
Turns the raw calendar columns into 0/1 flags for the target date. Holiday names are not kept, because the test holidays never appear in train:
- is_holiday: national holiday.
- is_cuti_bersama: collective leave day.
- is_school_holiday: school break (DKI Jakarta).

In [11]:
full["is_holiday"] = (full["holiday_tipe"] == "holiday").astype(int)
full["is_cuti_bersama"] = full["cuti_bersama_name"].notna().astype(int)
full["is_school_holiday"] = full["school_break"].notna().astype(int)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

,is_holiday,is_cuti_bersama,is_school_holiday
is_train,,,
0,0.041,0.033,0.173
1,0.054,0.018,0.114


### Genre columns
A film can have several genres ("Action, War"), so genre becomes one 0/1 column per genre.

In [12]:
genre_dummies = full["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
genre_dummies.columns = genre_dummies.columns.str.replace(" ", "_")
full = pd.concat([full, genre_dummies], axis=1)
GENRE_COLS = list(genre_dummies.columns)
print(f"genres: {len(GENRE_COLS)}")
genre_dummies.loc[full["is_train"] == 1].mean().sort_values(ascending=False).round(3).head(10)

genres: 27


genre_Horror       0.375
genre_Drama        0.328
genre_Action       0.212
genre_Thriller     0.146
genre_Comedy       0.120
genre_Animation    0.070
genre_Mystery      0.063
genre_Fantasy      0.056
genre_Superhero    0.052
genre_Family       0.043
dtype: float64

### Category columns
Columns with a small fixed set of text values become pandas categories, which LightGBM reads directly. Train and test share one table, so they get the same categories.

In [13]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
for col in CAT_COLS:
    full[col] = full[col].astype("category")
full[CAT_COLS].nunique()

city_name     69
day_tipe       3
age_rating     4
dtype: int64

### Drop columns without signal
Removes raw columns that are replaced by the flags and genre columns above, helper columns used only for the joins, and free-text credits (producer, director, writer, casts), since most names in test films never appear in train. ramadan_day is kept for postprocessing only.

In [14]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]
full = full.drop(columns=DROP_COLS)
print(f"full: {full.shape}")
print(list(full.columns))

full: (127282, 52)
['movie_title', 'cinema_ids', 'd1_date', 't1', 't2', 't3', 'occ1', 'occ2', 'occ3', 'show1', 'show2', 'show3', 'horizon', 'date_show', 'total_ticket', 'is_train', 'id', 'city_name', 'day_tipe', 'price', 'age_rating', 'ramadan_day', 'is_holiday', 'is_cuti_bersama', 'is_school_holiday', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Disaster', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Gothic', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_Religi', 'genre_Romance', 'genre_Sci-Fi', 'genre_Sport', 'genre_Superhero', 'genre_Survival', 'genre_Suspense', 'genre_Teen', 'genre_Thriller', 'genre_War']


### Target
The target is tickets divided by the pair's D1-D3 average (clipped at 1, the same scale the MASE metric uses), so a model with an L1 objective directly optimizes the competition metric.

In [15]:
full["scale"] = full[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
full["target_ratio"] = full["total_ticket"] / full["scale"]
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

horizon
4     0.915
5     0.616
6     0.467
7     0.360
8     0.163
9     0.000
10    0.000
Name: target_ratio, dtype: float64

# Feature Engineering

### Days-off blocks
Holidays differ mostly by how many days off they create in a row: a holiday on Thursday with a cuti bersama Friday makes a 4-day weekend, while a holiday on a Sunday adds almost nothing. A day counts as off if it is a weekend, a national holiday (holidays.csv plus the two holidays just before it from external/) or a cuti bersama day. The calendar starts 30 days before the first D1 so that blocks crossing the start of the data keep their full length:
- off_block_len: length of the block of consecutive days off that contains the target date (0 on a working day).

D1-D3 versions of this feature were tried and removed: in train, films opening on a long weekend drop hard afterwards, so the model predicted a steep drop for the Lebaran 2026 films, whose demand actually stays high through Eid week (LB 0.48785 vs 0.47566).

In [16]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]

full["off_block_len"] = full["date_show"].map(BLOCK_LEN)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

longest blocks:
         min        max  size
0 2025-03-28 2025-04-07    11
1 2025-05-10 2025-05-13     4
2 2025-05-29 2025-06-01     4
3 2025-06-06 2025-06-09     4
4 2025-12-25 2025-12-28     4
5 2026-02-14 2026-02-17     4
6 2026-03-18 2026-03-24     7


off_block_len,0,1,2,3,4,7,11
is_train,,,,,,,
0,0.658,0.005,0.228,0.019,0.047,0.042,NaN
1,0.657,0.006,0.213,0.072,0.051,NaN,0.001


### Pair features
Features from the pair's own D1-D3 sales:
- trend: D3 tickets compared with D1, to see whether sales rise or fall.
- t3_share: share of D3 in the D1-D3 total.
- occ_mean, show_mean: average occupancy and number of shows over D1-D3.
- tickets_per_show3: tickets per show on D3.
- active_days: how many of D1-D3 had sales.

In [17]:
D13 = ["t1", "t2", "t3"]
full["trend"] = (full["t3"] + 1) / (full["t1"] + 1)
full["t3_share"] = full["t3"] / full[D13].sum(axis=1)
full["occ_mean"] = full[["occ1", "occ2", "occ3"]].mean(axis=1)
full["show_mean"] = full[["show1", "show2", "show3"]].mean(axis=1)
full["tickets_per_show3"] = full["t3"] / full["show3"].replace(0, np.nan)
full["active_days"] = (full[D13] > 0).sum(axis=1)
full[["trend", "t3_share", "occ_mean", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

,trend,t3_share,occ_mean,show_mean,tickets_per_show3,active_days
count,127282.00,127282.00,127282.00,127282.00,127282.00,127282.00
mean,11.13,0.33,19.76,8.34,32.42,2.90
std,51.19,0.16,16.83,11.10,31.17,0.36
min,0.01,0.00,0.00,0.33,0.25,1.00
25%,0.56,0.24,7.87,3.33,12.14,3.00
50%,0.89,0.32,14.80,5.00,24.00,3.00
75%,1.49,0.40,26.28,9.00,42.20,3.00
max,1199.00,1.00,100.00,216.00,421.33,3.00


### Film features
National D1-D3 sales of the film over all its cinemas, because a film rises and falls in a similar way everywhere, and the LB showed big and rising films hold up better in the test period. Train and test rows are computed from their own windows, since a few titles appear in both (previews in train):
- mv_t1, mv_t2, mv_t3: national tickets per day.
- mv_trend: national D3 compared with D1.
- mv_clusters: number of cinemas still selling on D3.
- pair_share: the pair's share of the film's national D1-D3 tickets.

In [18]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]
for flag, window in [(1, train_window), (0, test_window)]:
    feats = film_features(window)
    rows = full["is_train"] == flag
    for col in FILM_COLS:
        full.loc[rows, col] = full.loc[rows, "movie_title"].map(feats[col]).values
full["pair_share"] = full[D13].sum(axis=1) / full[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

,mv_t1,mv_t2,mv_t3,mv_clusters,mv_trend,pair_share
is_train,,,,,,
0,11615.0,12300.0,12249.0,87.0,0.869,0.008
1,22042.0,18511.0,17704.0,81.0,0.923,0.010


### Cinema features
Average D1-D3 scale of each cinema over all films in train and test_history, as a measure of how big the cinema is that does not use any D4-D10 target:
- cl_scale: average pair scale of the cinema.
- cl_movies: number of films the cinema showed.

In [19]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)
full = full.join(cinema_feats, on="cinema_ids")
cinema_feats.describe().round(2)

,cl_scale,cl_movies
count,121.00,121.00
mean,192.95,192.13
std,165.41,73.64
min,18.77,5.00
25%,106.63,145.00
50%,160.53,189.00
75%,221.98,239.00
max,1348.26,347.00


### Weekday and price features
- dow, d1_dow: day of week of the target date and of D1, since where the weekend falls inside D1-D10 shifts the curve.
- price_ratio: ticket price on the target date compared with the average price over D1-D3.

In [20]:
full["dow"] = full["date_show"].dt.dayofweek
full["d1_dow"] = full["d1_date"].dt.dayofweek

DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def price_on(dates):
    keys = pd.MultiIndex.from_arrays([full["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
    return pd.Series(PRICE_MAP.reindex(keys).values, index=full.index)

d13_price = pd.concat([price_on(full["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
full["price_ratio"] = full["price"] / d13_price
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

,dow,d1_dow,price_ratio
count,127282.0,127282.000,127282.000
mean,3.0,2.772,0.977
std,2.0,0.726,0.144
min,0.0,0.000,0.615
25%,1.0,2.000,0.875
50%,3.0,3.000,0.955
75%,5.0,3.000,1.105
max,6.0,6.000,1.500


### Cinepoint features (trial / assumption)
Real national daily admissions of each film from cinepoint.com (daily top box office), scraped into `external/cinepoint/`. Train period (Apr-Sep 2025) is before the cutoff; the test period (Oct 2025 - Mar 2026) is after it, so models using these columns are a "trial" (see logs.md).

Per film, from its D1 date:
- `cp_log_m13`: log of the mean real admissions on D1-D3.
- `retK`: real admissions on day K divided by that mean (K = 4..10), empty when the film is not in the daily list that day.
- `cutK`: smallest admissions in the list on that date divided by the same mean. When the film is missing from the list, its real sales were below this.
- `cp_cov`: our D1-D3 tickets over the real D1-D3 admissions, how much of the national sales our data sees.
- `ret_h`, `cut_h`: `retK` / `cutK` at the row's own horizon. `cp_ret_mean`, `cp_ndays`: mean of the known `retK` and number of known days.

Train and test use the same code, so the model sees the same kind of signal in both. Set `USE_CINEPOINT = False` to switch the columns off.

In [21]:
USE_CINEPOINT = True
CP_FORMAT = r"\s*\((?:IMAX[^)]*|3D|2D|4DX)\)\s*$"
CP_HORIZONS = list(range(4, 11))
CP_FEATURES = (["cp_log_m13", "cp_cov", "cp_ndays", "ret_h", "cut_h", "cp_ret_mean"]
               + [f"ret{k}" for k in CP_HORIZONS] + [f"cut{k}" for k in CP_HORIZONS])
cp_key = lambda s: re.sub(r"[^a-z0-9]", "", re.sub(CP_FORMAT, "", s, flags=re.I).lower())

cp_files = [f"{EXTERNAL_DIR}/cinepoint/{n}" for n in ("cinepoint_daily_top_train.csv", "cinepoint_daily_top.csv")]
cp = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in cp_files if os.path.exists(f)], ignore_index=True)
cp["key"] = cp["title"].map(cp_key)
CP_CUT = cp.groupby("date")["daily_admission"].min()
CP_DAILY = {k: g.groupby("date")["daily_admission"].sum() for k, g in cp.groupby("key")}

def cp_film_features(rows):
    pairs = rows.drop_duplicates(["movie_title", "cinema_ids"])
    films = pairs.groupby("movie_title").agg(d1=("d1_date", "min"), d13=("t1", "sum"), t2=("t2", "sum"), t3=("t3", "sum"))
    films["d13"] += films["t2"] + films["t3"]
    out = []
    for title, f in films.iterrows():
        rec = {"movie_title": title}
        daily = CP_DAILY.get(cp_key(title))
        days = pd.date_range(f["d1"], periods=10)
        real = daily.reindex(days) if daily is not None else pd.Series(np.nan, index=days)
        if real.iloc[:3].notna().any():
            m13 = real.iloc[:3].mean()
            rec.update(cp_log_m13=np.log1p(m13), cp_cov=f["d13"] / (3 * m13), cp_ndays=int(real.iloc[3:].notna().sum()))
            for k in CP_HORIZONS:
                rec[f"ret{k}"] = real.iloc[k - 1] / m13
                rec[f"cut{k}"] = CP_CUT.get(days[k - 1], np.nan) / m13
        out.append(rec)
    return pd.DataFrame(out)

if USE_CINEPOINT:
    # train and test are done apart: a few titles occur in both with different D1 dates
    parts = []
    for flag in (1, 0):
        rows = full[full["is_train"] == flag]
        parts.append(rows.merge(cp_film_features(rows), on="movie_title", how="left"))
    full = pd.concat(parts, ignore_index=True)
    for col in CP_FEATURES:
        if col not in full:
            full[col] = np.nan
    idx = full["horizon"].to_numpy() - 4
    ret = full[[f"ret{k}" for k in CP_HORIZONS]].to_numpy(dtype=float)
    cut = full[[f"cut{k}" for k in CP_HORIZONS]].to_numpy(dtype=float)
    full["ret_h"] = ret[np.arange(len(full)), idx]
    full["cut_h"] = cut[np.arange(len(full)), idx]
    full["cp_ret_mean"] = np.nanmean(ret, axis=1)
    print(f"Cinepoint rows: {len(cp)}, dates {cp['date'].min().date()} to {cp['date'].max().date()}")
    display(full.groupby("is_train")[["cp_log_m13", "cp_cov", "cp_ndays", "ret_h", "cut_h"]].agg(lambda s: s.notna().mean()).round(3))

Cinepoint rows: 6034, dates 2025-04-01 to 2026-03-31


C:\Users\leouw\AppData\Local\Temp\ipykernel_23584\1083700288.py:48: RuntimeWarning: Mean of empty slice
  full["cp_ret_mean"] = np.nanmean(ret, axis=1)


,cp_log_m13,cp_cov,cp_ndays,ret_h,cut_h
is_train,,,,,
0,1.000,1.000,1.000,0.885,1.000
1,0.995,0.995,0.995,0.832,0.995


In [22]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,occ1,occ2,occ3,show1,...,cut7,ret8,cut8,ret9,cut9,ret10,cut10,ret_h,cut_h,cp_ret_mean
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,13.56,14.30,12.25,18.0,...,0.009174,0.288374,0.007454,0.206139,0.018014,0.32942,0.014335,1.408136,0.006737,0.667346
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,39.38,23.87,22.75,23.0,...,0.009174,0.288374,0.007454,0.206139,0.018014,0.32942,0.014335,1.408136,0.006737,0.667346
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.00,16.10,16.14,0.0,...,0.009174,0.288374,0.007454,0.206139,0.018014,0.32942,0.014335,1.408136,0.006737,0.667346
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,19.57,38.15,43.26,9.0,...,0.009174,0.288374,0.007454,0.206139,0.018014,0.32942,0.014335,1.408136,0.006737,0.667346
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,12.21,14.42,15.14,30.0,...,0.009174,0.288374,0.007454,0.206139,0.018014,0.32942,0.014335,1.408136,0.006737,0.667346


# Regressor Models

### Setup and validation
All models predict target_ratio (tickets / D1-D3 average), so the mean absolute error on it is exactly the competition MASE. Validation is GroupKFold by film, so a validation film is never seen in training, like the test films. This CV is reliable for model and parameter choices, but not for anything tied to the test season, which only the LB can judge. Every model is then retrained on 100% of train to predict test.
- RUN_MODELS: which model cells actually train; the others are skipped.
- SEED: one constant for every source of randomness, so the results can be reproduced.
- MODEL_SEEDS: the tree models are trained once per seed and the predictions averaged with equal weights (never a seed picked by its CV score).
- ZERO_SNAP: tree models predict a small positive number where the true median is 0 (31.8% of train targets are exactly 0), so predictions below the threshold are set to 0. Out of fold this improved LightGBM from 0.3727 to 0.3673 and CatBoost from 0.3704 to 0.3653, also when the threshold is chosen on four folds and tested on the fifth.

In [23]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["lgbm", "catboost", "kumo"]
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2, "catboost": 0.2, "xgboost": 0.2}
N_FOLDS = 5

GENRE_COLS = [c for c in full.columns if c.startswith("genre_")]
NUM_FEATURES = [
    "t1", "t2", "t3", "occ1", "occ2", "occ3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "occ_mean", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
] + GENRE_COLS + (CP_FEATURES if USE_CINEPOINT else [])
FEATURES = NUM_FEATURES + CAT_COLS

train_df = full[full["is_train"] == 1].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
X, y, groups = train_df[FEATURES], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES]
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))

OOF, TEST_PRED, SCORES = {}, {}, {}

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d):
    preds = [make_model(seed).fit(prep(X_tr), y_tr).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

def cross_validate(name, fit_predict):
    snap = lambda p: np.where(p < ZERO_SNAP.get(name, 0.0), 0.0, p)
    oof = np.zeros(len(train_df))
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        oof[va_idx] = snap(np.clip(fit_predict(X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx]), 0, None))
        fold_scores.append(mase_ratio(y[va_idx], oof[va_idx]))
        print(f"{name} fold {fold}: MASE={fold_scores[-1]:.4f}")
    OOF[name], SCORES[name] = oof, fold_scores
    print(f"{name} CV MASE: {mase_ratio(y, oof):.4f}")
    TEST_PRED[name] = snap(np.clip(fit_predict(X, y, X_test), 0, None))

print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | test rows: {len(test_df)}")
print(f"naive predict 0       : {mase_ratio(y, 0):.4f}")
print(f"naive predict D1-D3 avg: {mase_ratio(y, 1):.4f}")

features: 83 | train rows: 54671 | test rows: 72611
naive predict 0       : 0.5936
naive predict D1-D3 avg: 0.7388


### LGBM Regressor
LightGBM with an L1 (absolute error) objective and the parameters found by Optuna on 2026-10-07 (71 trials, 5 folds grouped by film, top trials re-scored on 3 seeds). Default parameters gave 0.3844 and these give 0.3725 averaged over 10 seeds.

In [24]:
LGBM_PARAMS = dict(
    n_estimators=883,
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, X_pred):
    return seed_average(lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS), X_tr, y_tr, X_pred)

if "lgbm" in RUN_MODELS:
    cross_validate("lgbm", lgbm_fit_predict)

lgbm fold 0: MASE=0.2649
lgbm fold 1: MASE=0.4528
lgbm fold 2: MASE=0.2159
lgbm fold 3: MASE=0.3363
lgbm fold 4: MASE=0.2547
lgbm CV MASE: 0.3049


### CatBoost Regressor
CatBoost with loss MAE and the Optuna parameters of 2026-10-07 (20 trials, top trials re-scored on 3 seeds). Category columns are passed as text, which CatBoost encodes itself. Default parameters gave 0.3791 and these give 0.3704 averaged over 3 seeds. It is the slowest model: each fit takes 1 to 3 minutes.

In [25]:
CATBOOST_PARAMS = dict(
    iterations=723,
    learning_rate=0.0329941,
    depth=10,
    l2_leaf_reg=11.9474,
    random_strength=7.56829,
    bagging_temperature=1.78965,
)

def catboost_fit_predict(X_tr, y_tr, X_pred):
    to_text = lambda df: df.astype({c: str for c in CAT_COLS})
    make_model = lambda seed: CatBoostRegressor(loss_function="MAE", random_seed=seed, verbose=0, cat_features=CAT_COLS, thread_count=-1, **CATBOOST_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, prep=to_text)

if "catboost" in RUN_MODELS:
    cross_validate("catboost", catboost_fit_predict)

catboost fold 0: MASE=0.2611
catboost fold 1: MASE=0.4618
catboost fold 2: MASE=0.2173
catboost fold 3: MASE=0.3549
catboost fold 4: MASE=0.2549
catboost CV MASE: 0.3100


### XGBoost Regressor
XGBoost with an absolute-error objective and the Optuna parameters of 2026-10-07 (60 trials, top trials re-scored on 3 seeds), reading the category columns directly. Default parameters gave 0.4047 and these give 0.3781 averaged over 10 seeds, still the weakest of the three, so it is off by default (add "xgboost" to RUN_MODELS to use it).

In [26]:
XGB_PARAMS = dict(
    n_estimators=541,
    learning_rate=0.0145094,
    max_depth=10,
    min_child_weight=46.1445,
    subsample=0.514938,
    colsample_bytree=0.431351,
    reg_lambda=7.78404,
    reg_alpha=2.25023,
)

def xgboost_fit_predict(X_tr, y_tr, X_pred):
    make_model = lambda seed: XGBRegressor(objective="reg:absoluteerror", enable_categorical=True, tree_method="hist", random_state=seed, **XGB_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred)

if "xgboost" in RUN_MODELS:
    cross_validate("xgboost", xgboost_fit_predict)

### Kumo-Tabular
NVIDIA's pretrained tabular foundation model (nvidia/Kumo-Tabular, large). It does not train: a random sample of training rows is passed as context and it predicts 999 quantiles per row, of which the median (quantile 0.5) is used because MASE rewards the median. It was the best model in CV by a wide margin (0.3447 against 0.3704 for the best tree model), partly because it predicts exact zeros where the median is 0, and partly because it generalizes better to films it has not seen. GPU memory grows with the context, so it is sampled:
- KUMO_CONTEXT_ROWS: training rows used as context per run (10,000 uses about 5 GB on an 8 GB GPU).
- KUMO_CONTEXT_SEEDS: one run per seed with a different context sample, averaged, so more of the training data is seen. Out-of-fold MAE was 0.3502 for 5,000 rows and 1 seed, 0.3458 for 5,000 rows and 3 seeds, 0.3465 for 10,000 rows and 1 seed and 0.3447 for 10,000 rows and 3 seeds. Three seeds at 10,000 rows take about 20 minutes in total on the RTX 4070.

In [27]:
KUMO_CONTEXT_ROWS = 10000
KUMO_CONTEXT_SEEDS = [42, 7, 123]
KUMO_STYPES = {c: "numerical" for c in NUM_FEATURES} | {c: "categorical" for c in CAT_COLS}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kumo_model = sdm.models.KumoTabular(task="regression", size="large", device=DEVICE).eval() if "kumo" in RUN_MODELS else None

def kumo_fit_predict(X_tr, y_tr, X_pred, chunk_rows=2000):
    preds = []
    for seed in KUMO_CONTEXT_SEEDS:
        ctx = np.random.default_rng(seed).choice(len(X_tr), min(KUMO_CONTEXT_ROWS, len(X_tr)), replace=False)
        x_ctx = TableTensor.from_pandas(X_tr.iloc[ctx], KUMO_STYPES, device=DEVICE)
        y_ctx = TableTensor.from_pandas(pd.DataFrame({"y": y_tr[ctx]}), {"y": "numerical"}, device=DEVICE)
        kumo_model.fit(x_ctx, y_ctx, num_estimators=4, generator=torch.Generator(device=DEVICE).manual_seed(seed))
        quantiles = []
        for start in range(0, len(X_pred), chunk_rows):
            x_q = TableTensor.from_pandas(X_pred.iloc[start:start + chunk_rows], KUMO_STYPES, device=DEVICE)
            quantiles.append(kumo_model.predict(x_q).numerical.float().cpu())
        q = torch.cat(quantiles).numpy()
        preds.append(q[:, q.shape[1] // 2])
        torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

if "kumo" in RUN_MODELS:
    cross_validate("kumo", kumo_fit_predict)

C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\tensor\string.py:502: RuntimeWarning: Falling back to a CPU-based string sort because cuDF is not installed. Install cuDF to enable faster CUDA-based string sorting without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(
C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\relational\join.py:73: RuntimeWarning: Falling back to a CPU-based join because cuDF is not installed. Install cuDF to enable faster CUDA-based joins without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(


kumo fold 0: MASE=0.2518
kumo fold 1: MASE=0.4368
kumo fold 2: MASE=0.2019
kumo fold 3: MASE=0.3188
kumo fold 4: MASE=0.2438
kumo CV MASE: 0.2906


### Model comparison
CV MASE of every model that ran, overall and per fold.

In [28]:
comparison = pd.DataFrame(
    {name: [mase_ratio(y, OOF[name])] + SCORES[name] for name in OOF},
    index=["CV"] + [f"fold {k}" for k in range(N_FOLDS)],
).T.sort_values("CV")
comparison.round(4)

,CV,fold 0,fold 1,fold 2,fold 3,fold 4
kumo,0.2906,0.2518,0.4368,0.2019,0.3188,0.2438
lgbm,0.3049,0.2649,0.4528,0.2159,0.3363,0.2547
catboost,0.3100,0.2611,0.4618,0.2173,0.3549,0.2549


### Blend
Weighted average of the models' predictions, a simple blend instead of a stacked meta-model because a meta-model would fit train-period patterns that may not hold in the test period.
- BLEND_WEIGHTS: weight of each model; models that did not run are skipped and the rest are rescaled to sum to 1.
- Out of fold, Kumo alone (0.3447) beats every blend of it with the tree models, and each 0.05 of weight moved to the tree models costs about 0.0004. The default keeps 0.8 on Kumo and 0.1 each on LightGBM and CatBoost (0.3466 in CV) as a hedge, because a Kumo-only submission has never been scored on the leaderboard. XGBoost is left out (weight 0).

In [29]:
BLEND_WEIGHTS = {"kumo": 0.8, "catboost": 0.1, "lgbm": 0.1, "xgboost": 0.0}
used = {m: w for m, w in BLEND_WEIGHTS.items() if m in OOF and w > 0}
total = sum(used.values())
oof_blend = sum(w * OOF[m] for m, w in used.items()) / total
test_blend = sum(w * TEST_PRED[m] for m, w in used.items()) / total
print(f"blend {({m: round(w / total, 3) for m, w in used.items()})}: CV MASE {mase_ratio(y, oof_blend):.4f}")
train_df.assign(err=np.abs(y - oof_blend)).groupby("horizon")["err"].mean().round(4)

blend {'kumo': 0.8, 'catboost': 0.1, 'lgbm': 0.1}: CV MASE 0.2911


horizon
4     0.3444
5     0.3678
6     0.2898
7     0.2917
8     0.2429
9     0.2486
10    0.2516
Name: err, dtype: float64

# Postprocessing

### Seasonal adjustment
Train starts on Idulfitri 2025, so the model never saw a film whose D1-D3 is at the end of Ramadan and whose D4-D10 is in Lebaran week. These factors were set on the public LB (0.47566 with the baseline model, 0.45395 with main.ipynb and default parameters):
- lebaran: D1-D3 touches Ramadan and the target date is from 20 March 2026 on, x2.0.
- libur_sekolah: D1-D3 and the target date are all school holidays, x1.25.

In [30]:
RAMADAN_START, LEBARAN_START = ramadan["date"].min(), pd.Timestamp("2026-03-20")
SCHOOL_DAYS = pd.DatetimeIndex(school["date"])
SEASON_FACTORS = {"lebaran": 2.0, "libur_sekolah": 1.25}

def season_factor(df):
    d13 = [df["d1_date"] + pd.Timedelta(days=k) for k in range(3)]
    d13_ramadan = np.any([(d >= RAMADAN_START) & (d < LEBARAN_START) for d in d13], axis=0)
    lebaran = d13_ramadan & (df["date_show"] >= LEBARAN_START)
    school_break = np.all([d.isin(SCHOOL_DAYS) for d in d13], axis=0) & df["date_show"].isin(SCHOOL_DAYS)
    return np.select([lebaran, school_break], [SEASON_FACTORS["lebaran"], SEASON_FACTORS["libur_sekolah"]], 1.0)

test_df["season_factor"] = season_factor(test_df)
test_df["total_ticket"] = test_blend * test_df["scale"] * test_df["season_factor"]
test_df["season_factor"].value_counts()

season_factor
1.00    64746
2.00     4598
1.25     3267
Name: count, dtype: int64

# Submission

In [31]:
def write_submission(ratio, path):
    out = test_df[["id"]].astype({"id": int}).assign(total_ticket=ratio * test_df["scale"].values * test_df["season_factor"].values)
    sub = sample_sub[["id"]].merge(out, on="id", how="left")
    assert len(sub) == len(sample_sub) and sub["total_ticket"].notna().all()
    sub.to_csv(path, index=False)
    return sub

submission = write_submission(test_blend, "submission.csv")
if "kumo" in TEST_PRED:
    write_submission(TEST_PRED["kumo"], "submission_kumo_only.csv")
trees = [m for m in ("lgbm", "catboost", "xgboost") if m in TEST_PRED]
if trees:
    write_submission(np.mean([TEST_PRED[m] for m in trees], axis=0), "submission_trees_only.csv")
submission.describe()

,id,total_ticket
count,72611.000000,72611.000000
mean,36306.000000,193.879352
std,20961.134535,743.835193
min,1.000000,0.000000
25%,18153.500000,0.018987
50%,36306.000000,26.045542
75%,54458.500000,145.577553
max,72611.000000,45774.774811
